# 26 — ICWS Weighted MinHash 512

Weighted MinHash / Consistent Weighted Sampling baseline for Weighted Jaccard. This produces a 512-sample sketch and ranks by sketch collision rate.

This is the most important non-neural Weighted Jaccard baseline for your paper.


In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np

sys.path.append('/raid/ruban/hpmlproj/term_project/SigSpatial')
from sota_experiment_common import cleanup, eval_recall, load_dataset, save_result

# Edit here
dataset_name = "full"       # ICWS brute sketch ranking is intended for 10k first
num_samples = 512
seed = 42
top_k = 500
OUT_PATH = "/tmp/results_sota_icws_512.pkl"
METHOD_NAME = "icws_weighted_minhash_512"
NOTEBOOK_NAME = "26_icws_weighted_minhash_512.ipynb"
np.random.seed(seed)


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [2]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
max_k = top_k


dataset=full | qt=(233773, 18220) | corpus=(187019, 18220) | queries=(46754, 18220)


In [3]:
# Ioffe-style Improved Consistent Weighted Sampling.
# Stores (dimension index, t) pairs; collision fraction estimates Weighted Jaccard.
def icws_signatures(x, num_samples=512, seed=42, batch_size=128):
    rng = np.random.default_rng(seed)
    d = x.shape[1]
    r = rng.gamma(shape=2.0, scale=1.0, size=(num_samples, d)).astype(np.float32)
    c = rng.gamma(shape=2.0, scale=1.0, size=(num_samples, d)).astype(np.float32)
    beta = rng.random(size=(num_samples, d), dtype=np.float32)

    sig_idx = np.empty((len(x), num_samples), dtype=np.int32)
    sig_t = np.empty((len(x), num_samples), dtype=np.int32)
    for start in range(0, len(x), batch_size):
        xb = np.maximum(x[start:start + batch_size], 1e-30).astype(np.float32)
        logx = np.log(xb)
        b = len(xb)
        idx_out = np.empty((b, num_samples), dtype=np.int32)
        t_out = np.empty((b, num_samples), dtype=np.int32)
        for s in range(num_samples):
            t = np.floor(logx / r[s] + beta[s]).astype(np.int32)
            y = np.exp(r[s] * (t - beta[s]))
            a = c[s] / (y * np.exp(r[s]))
            idx = np.argmin(a, axis=1)
            idx_out[:, s] = idx
            t_out[:, s] = t[np.arange(b), idx]
        sig_idx[start:start + b] = idx_out
        sig_t[start:start + b] = t_out
        print(f"signed {start + b:,}/{len(x):,}", flush=True)
    return sig_idx, sig_t

t0 = time.time()
sig_idx, sig_t = icws_signatures(qt, num_samples=num_samples, seed=seed)
print(f"signature time={(time.time()-t0)/60:.1f} min")
corpus_idx, query_idx = sig_idx[:query_start], sig_idx[query_start:]
corpus_t, query_t = sig_t[:query_start], sig_t[query_start:]
print(f"signature memory corpus={(corpus_idx.nbytes + corpus_t.nbytes)/1024**2:.1f} MB")


signed 128/233,773
signed 256/233,773
signed 384/233,773
signed 512/233,773
signed 640/233,773
signed 768/233,773
signed 896/233,773
signed 1,024/233,773
signed 1,152/233,773
signed 1,280/233,773
signed 1,408/233,773
signed 1,536/233,773
signed 1,664/233,773
signed 1,792/233,773
signed 1,920/233,773
signed 2,048/233,773
signed 2,176/233,773
signed 2,304/233,773
signed 2,432/233,773
signed 2,560/233,773
signed 2,688/233,773
signed 2,816/233,773
signed 2,944/233,773
signed 3,072/233,773
signed 3,200/233,773
signed 3,328/233,773
signed 3,456/233,773
signed 3,584/233,773
signed 3,712/233,773
signed 3,840/233,773
signed 3,968/233,773
signed 4,096/233,773
signed 4,224/233,773
signed 4,352/233,773
signed 4,480/233,773
signed 4,608/233,773
signed 4,736/233,773
signed 4,864/233,773
signed 4,992/233,773
signed 5,120/233,773
signed 5,248/233,773
signed 5,376/233,773
signed 5,504/233,773
signed 5,632/233,773
signed 5,760/233,773
signed 5,888/233,773
signed 6,016/233,773
signed 6,144/233,773
signed

In [4]:
def sketch_topk(query_idx, query_t, corpus_idx, corpus_t, k=500, batch_size=16):
    nbrs = []
    t0 = time.time()
    for start in range(0, len(query_idx), batch_size):
        qi = query_idx[start:start + batch_size]
        qt_ = query_t[start:start + batch_size]
        sim = ((qi[:, None, :] == corpus_idx[None, :, :]) &
               (qt_[:, None, :] == corpus_t[None, :, :])).mean(axis=2)
        kk = min(k, corpus_idx.shape[0])
        part = np.argpartition(-sim, kk - 1, axis=1)[:, :kk]
        rows = np.arange(len(qi))[:, None]
        order = np.argsort(-sim[rows, part], axis=1)
        top = part[rows, order]
        nbrs.extend([row.tolist() for row in top])
        print(f"ranked {start + len(qi):,}/{len(query_idx):,}", flush=True)
    qps = len(query_idx) / max(time.time() - t0, 1e-9)
    return nbrs, qps

nbrs, qps = sketch_topk(query_idx, query_t, corpus_idx, corpus_t, k=top_k)
metrics = {**eval_recall(gt, nbrs, query_start, top_k), "qps": qps, "samples": num_samples}
for k, v in metrics.items():
    if isinstance(k, int):
        print(f"R@{k:<4} = {v:.4f}")
print(f"QPS={qps:.1f}")
save_result(OUT_PATH, dataset_name, METHOD_NAME, metrics, meta={"notebook": NOTEBOOK_NAME})
cleanup()


ranked 16/46,754
ranked 32/46,754
ranked 48/46,754
ranked 64/46,754
ranked 80/46,754
ranked 96/46,754
ranked 112/46,754
ranked 128/46,754
ranked 144/46,754
ranked 160/46,754
ranked 176/46,754
ranked 192/46,754
ranked 208/46,754
ranked 224/46,754
ranked 240/46,754
ranked 256/46,754
ranked 272/46,754
ranked 288/46,754
ranked 304/46,754
ranked 320/46,754
ranked 336/46,754
ranked 352/46,754
ranked 368/46,754
ranked 384/46,754
ranked 400/46,754
ranked 416/46,754
ranked 432/46,754
ranked 448/46,754
ranked 464/46,754
ranked 480/46,754
ranked 496/46,754
ranked 512/46,754
ranked 528/46,754
ranked 544/46,754
ranked 560/46,754
ranked 576/46,754
ranked 592/46,754
ranked 608/46,754
ranked 624/46,754
ranked 640/46,754
ranked 656/46,754
ranked 672/46,754
ranked 688/46,754
ranked 704/46,754
ranked 720/46,754
ranked 736/46,754
ranked 752/46,754
ranked 768/46,754
ranked 784/46,754
ranked 800/46,754
ranked 816/46,754
ranked 832/46,754
ranked 848/46,754
ranked 864/46,754
ranked 880/46,754
ranked 896/46,75